In [ ]:
!pip install -q insightface onnxruntime opencv-python scikit-learn pandas
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving person_01_0.jpg to person_01_0.jpg
Saving person_01_1.jpg to person_01_1.jpg
Saving person_02_0.jpg to person_02_0.jpg
Saving person_02_1.jpg to person_02_1.jpg
Saving person_03_0.jpg to person_03_0.jpg
Saving person_03_1.jpg to person_03_1.jpg


In [ ]:
import cv2
import numpy as np
import pandas as pd
from insightface.app import FaceAnalysis

# -----------------------
# Load InsightFace Model
# -----------------------

app = FaceAnalysis(name="buffalo_l")
app.prepare(
    ctx_id=-1,
    det_size=(1024, 1024)
)

embeddings = []
image_names = []

# -----------------------
# Process Images
# -----------------------

for file in uploaded.keys():

    img = cv2.imread(file)

    if img is None:
        print("Cannot read:", file)
        continue

    faces = app.get(img)

    print(f"{file} -> Faces Detected: {len(faces)}")

    if len(faces) == 0:
        continue

    # Select face nearest to image center
    h, w = img.shape[:2]
    cx, cy = w / 2, h / 2

    best_face = min(
        faces,
        key=lambda f: (
            ((f.bbox[0] + f.bbox[2]) / 2 - cx) ** 2 +
            ((f.bbox[1] + f.bbox[3]) / 2 - cy) ** 2
        )
    )

    emb = best_face.embedding.astype(np.float32)
    emb = emb / np.linalg.norm(emb)

    embeddings.append(emb)
    image_names.append(file)

embeddings = np.array(embeddings)

print("Total valid faces:", len(embeddings))

Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/w600k_r50.onnx recognition ['None', 3, 112, 112] 127.5 127.5
set det-size: (1024, 1024)
person_01_

In [ ]:
from sklearn.cluster import AgglomerativeClustering

# Number of people in the dataset
NUM_PEOPLE = 3

clustering = AgglomerativeClustering(
    n_clusters=NUM_PEOPLE,
    metric="cosine",
    linkage="average"
)

labels = clustering.fit_predict(embeddings)

print("Cluster Labels:", labels)

Cluster Labels: [2 2 0 0 1 1]


In [ ]:
for cluster in np.unique(labels):
    idx = np.where(labels == cluster)[0]

    print(f"\nCluster {cluster}")

    for i in idx:
        for j in idx:
            if i != j:
                sim = cosine_similarity(
                    embeddings[i].reshape(1, -1),
                    embeddings[j].reshape(1, -1)
                )[0][0]

                print(image_names[i], "<->", image_names[j], "=", sim)


Cluster 0
person_02_0.jpg <-> person_02_1.jpg = 0.5759735
person_02_1.jpg <-> person_02_0.jpg = 0.5759735

Cluster 1
person_03_0.jpg <-> person_03_1.jpg = 0.7268454
person_03_1.jpg <-> person_03_0.jpg = 0.7268454

Cluster 2
person_01_0.jpg <-> person_01_1.jpg = 0.7071424
person_01_1.jpg <-> person_01_0.jpg = 0.7071424


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd

results = []

for cluster in np.unique(labels):

    idx = np.where(labels == cluster)[0]

    for i in idx:

        similarities = []

        for j in idx:
            if i != j:
                sim = cosine_similarity(
                    embeddings[i].reshape(1, -1),
                    embeddings[j].reshape(1, -1)
                )[0][0]
                similarities.append(sim)

        if len(similarities) == 0:
            confidence = 100.0
        else:
            avg_sim = np.mean(similarities)

            # Convert similarity to confidence
            confidence = max(0, min(100, (avg_sim + 1) / 2 * 100))

        results.append([
            image_names[i],
            f"Person_{cluster+1}",
            round(confidence, 2)
        ])

df = pd.DataFrame(
    results,
    columns=["Image Name", "Cluster", "Confidence (%)"]
)

print(df)

df.to_csv("person_clusters.csv", index=False)

        Image Name   Cluster  Confidence (%)
0  person_02_0.jpg  Person_1       78.800003
1  person_02_1.jpg  Person_1       78.800003
2  person_03_0.jpg  Person_2       86.339996
3  person_03_1.jpg  Person_2       86.339996
4  person_01_0.jpg  Person_3       85.360001
5  person_01_1.jpg  Person_3       85.360001


In [ ]:
from google.colab import files

files.download("person_clusters.csv")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import shutil

os.makedirs("clusters", exist_ok=True)

for img_name, cluster in zip(image_names, labels):

    folder = os.path.join("clusters", f"Person_{cluster+1}")
    os.makedirs(folder, exist_ok=True)

    shutil.copy(img_name, os.path.join(folder, os.path.basename(img_name)))

In [ ]:
import shutil

shutil.make_archive("clusters", "zip", "clusters")

In [ ]:
from google.colab import files

files.download("clusters.zip")

In [ ]:
import pandas as pd

df = pd.read_csv("person_clusters.csv")
print(df)

In [ ]:
print(df.groupby("Cluster")["Image Name"].apply(list))


In [ ]:
print(df)

print("\nTotal Images :", len(df))
print("Total Clusters :", df["Cluster"].nunique())